In [26]:
import PyPDF2
import re
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [27]:
from src.config.db import get_connection

conn2 = get_connection()

conn2.autocommit = True

In [28]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

pdf_path = PROJECT_ROOT / "docs" / "2023-annual-report-truncated.pdf"

reader = PyPDF2.PdfReader(str(pdf_path))

In [29]:

def generate_document_id(conn2):
    with conn2.cursor() as cur:
        cur.execute("""
            SELECT id
            FROM "Document"
            ORDER BY id DESC
            LIMIT 1
        """)

        row = cur.fetchone()

    if row is None:
        return "DOC000001"

    last_number = int(re.search(r"\d+", row[0]).group())

    return f"DOC{last_number + 1:06d}"

In [30]:
document_id = generate_document_id(conn2)

document_title = pdf_path.stem
source_file = pdf_path.name
total_pages = len(reader.pages)

with conn2.cursor() as cur:

    cur.execute(
        """
        INSERT INTO "Document"
        (
            id,
            title,
            "sourceFile",
            "totalPages"
        )
        VALUES
        (%s,%s,%s,%s)
        """,
        (
            document_id,
            document_title,
            source_file,
            total_pages,
        ),
    )

conn2.commit()

print("Document Stored Successfully")
print(document_id)

Document Stored Successfully
DOC000001


In [31]:
# with conn.cursor() as cur:
#     cur.execute('SELECT COUNT(*) FROM "Document"')
#     # print(cur.fetchone())

In [32]:
with conn2.cursor() as cur:

    for page_number, page in enumerate(reader.pages, start=1):

        page_id = f"{document_id}_P{page_number:03d}"

        text = page.extract_text()

        if text is None:
            text = ""

        text = text.strip()

        token_count = len(text.split())

        cur.execute(
            """
            INSERT INTO "Page"
            (
                id,
                "documentId",
                "pageNumber",
                content,
                "tokenCount"
            )
            VALUES
            (%s,%s,%s,%s,%s)
            """,
            (
                page_id,
                document_id,
                page_number,
                text,
                token_count,
            ),
        )

conn2.commit()

print(f"{len(reader.pages)} Pages Stored Successfully")

50 Pages Stored Successfully


In [33]:
with conn2.cursor() as cur:

    cur.execute("""
        SELECT
            id,
            "pageNumber",
            "tokenCount"
        FROM "Page"
        WHERE "documentId"=%s
        ORDER BY "pageNumber"
    """, (document_id,))

    rows = cur.fetchall()

for row in rows[:5]:
    print(row)

('DOC000001_P001', 1, 30)
('DOC000001_P002', 2, 0)
('DOC000001_P003', 3, 260)
('DOC000001_P004', 4, 88)
('DOC000001_P005', 5, 118)
